<a href="https://colab.research.google.com/github/engmohammedhisham/machine-learning-intern/blob/main/work/notebooks/w03_data_contract_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents a unique piece of content (`content_id`) managed for a specific client (`client_id`).

**Time Window:** The performance metrics capture a rolling lookback window of **90 days** (e.g., `impressions_90d`, `clicks_90d`, `avg_position`), alongside short-term comparative windows tracking momentum over the **last 30 days** versus the **previous 30 days** to evaluate directional trend shifts.

In [3]:
import pandas as pd
df = pd.read_csv('content_refresh_anonymized.csv')
total_rows = len(df)
unique_contents = df['content_id'].nunique()
print(f"Total rows in dataset: {total_rows}")
print(f"Unique content IDs: {unique_contents}")
print(f"Is content_id the exact grain/unit of analysis? {total_rows == unique_contents}")

Total rows in dataset: 4681
Unique content IDs: 4681
Is content_id the exact grain/unit of analysis? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization Contract:

* **Features (المدخلات):**
  * `word_count`, `search_volume`, `competition_level` (Structural signals)
  * `impressions_90d`, `clicks_90d`, `ctr`, `avg_position` (Historical 90-day core performance)
  * `impressions_last_30d`, `clicks_last_30d`, `impressions_prev_30d`, `clicks_prev_30d` (Short-term velocity signals)

* **Label (الهدف):**
  * `trend_direction` (Specifically classifying if the article trend is `'down'` to detect content decay and flag it for a refresh).

* **Context (السياق):**
  * `content_type`, `main_intent`, `provider_used`, `model_used`, `age_tier` (Useful for performance slicing and post-mortem analysis).

* **Excluded (المستبعدة مع السبب):**
  * `content_id`, `client_id`: High-cardinality string hashes that do not contain statistical predictive value and cause severe model overfitting.
  * `trend_pct`: Excluded because it creates direct **Target Leakage** (it is mathematically derived using the target outcome we want to predict).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# 1. Verification of missing values in core contractual fields
print("### 1. Missing Values Audit ###")
missing_summary = df[['content_id', 'trend_direction', 'clicks_90d', 'avg_position', 'word_count']].isnull().sum()
print(missing_summary)

# 2. Verification of target label distribution
print("\n### 2. Label Distribution Analysis ###")
print(df['trend_direction'].value_counts(dropna=False))

# 3. Verification of statistical boundaries (Data Range Check)
print("\n### 3. Data Range Check (Statistical Summary) ###")
display(df[['word_count', 'clicks_90d', 'ctr', 'avg_position', 'search_volume']].describe())

### 1. Missing Values Audit ###
content_id            0
trend_direction       1
clicks_90d            0
avg_position          1
word_count         1225
dtype: int64

### 2. Label Distribution Analysis ###
trend_direction
down      2544
stable     946
up         672
new        340
flat       178
NaN          1
Name: count, dtype: int64

### 3. Data Range Check (Statistical Summary) ###


,word_count,clicks_90d,ctr,avg_position,search_volume
count,3456.000000,4681.000000,4680.000000,4680.000000,4291.000000
mean,3089.893519,16.042085,0.589564,16.118590,161.787462
std,1432.617330,71.277145,3.584444,15.114041,1414.431281
min,8.000000,0.000000,0.000000,0.000000,0.000000
25%,2421.750000,0.000000,0.000000,6.200000,0.000000
50%,2866.000000,1.000000,0.070000,10.400000,10.000000
75%,3662.000000,6.000000,0.280000,22.200000,20.000000
max,9059.000000,2138.000000,100.000000,138.800000,40500.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits & What this data can NEVER tell us:**

1. **Lack of Seasonality:** Because this dataset provides a static snapshot aggregating 90 days of metrics, it cannot distinguish between annual seasonal traffic drops (e.g., an "ice cream recipe" article losing traffic in December) and structural content decay.
2. **External Search Changes:** The metrics show an observed drop in performance (`avg_position` or `ctr`), but the data can never explicitly tell us *why* it dropped—whether it was due to a core Google Algorithm update, or because a direct competitor launched a vastly superior article.
3. **Downstream Actions:** While we can track visibility (`impressions_90d`) and engagement (`clicks_90d`), the contract lacks conversion and revenue markers; it cannot tell us if the users who clicked actually converted or bounded away instantly.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.